# Run the Streamlit Viewer

Use this notebook after Parts 2 and 3 have produced the B08 GeoTIFF and `autorift_scatter_demeaned_*.npz` files. It starts `4. autorift_speed_scatter_app.py` in Codespaces.

Keep this notebook in the same folder as `4. autorift_speed_scatter_app.py` and the Part 3 outputs.


## 1. Check viewer dependencies

The Codespaces environment should already include these packages. This cell checks that the viewer can import them.


In [1]:
from __future__ import annotations

import importlib.util

REQUIRED_MODULES = [
    "streamlit",
    "plotly",
    "rasterio",
    "PIL",
    "numpy",
    "matplotlib",
]
missing = [mod for mod in REQUIRED_MODULES if importlib.util.find_spec(mod) is None]
if missing:
    raise RuntimeError(
        "Missing viewer dependencies: "
        + ", ".join(missing)
        + ". In Codespaces, select the Python (rsa-lab) kernel or rebuild the container."
    )
print("Viewer dependencies are ready.")


Viewer dependencies are ready.


## 2. Check files

The Streamlit app needs the `.py` file, the Part 3 `.npz`, and preferably the Part 2 B08 GeoTIFF for the basemap.


In [2]:
from pathlib import Path

LAB_DIR = Path.cwd().resolve()
APP = LAB_DIR / "4. autorift_speed_scatter_app.py"
npz_files = sorted(LAB_DIR.glob("autorift_scatter_demeaned_*.npz"))
basemap = LAB_DIR / "B08_fox_2026-03-01.tif"

if not APP.is_file():
    raise FileNotFoundError(APP)
if not npz_files:
    raise FileNotFoundError("No autorift_scatter_demeaned_*.npz found. Run Part 3 first.")

print(f"Working folder: {LAB_DIR}")
print(f"App: {APP.name}")
print("NPZ files:")
for f in npz_files:
    print(" -", f.name)
print(f"Basemap: {basemap.name if basemap.is_file() else '(not found; app still runs without basemap)'}")


Working folder: /home/xguo/Remote-Sense-Application/FeatureTrackingLab
App: 4. autorift_speed_scatter_app.py
NPZ files:
 - autorift_scatter_demeaned_2026-03-01__2026-03-21.npz
Basemap: B08_fox_2026-03-01.tif


## 3. Start Streamlit

Run this cell, then open the forwarded `8501` port from the Codespaces **Ports** panel. The app keeps running while this cell is active. Stop it with the next cell when you are done.


In [3]:
import subprocess
import sys
import time
from pathlib import Path

PORT = 8501
LOG_FILE = Path("streamlit_viewer.log")

# Stop old Streamlit processes from a previous run in this same Codespace/session.
subprocess.run(["pkill", "-f", "streamlit run"], check=False)
time.sleep(1)

streamlit_log = LOG_FILE.open("w")
streamlit_proc = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        str(APP),
        "--server.port",
        str(PORT),
        "--server.address",
        "0.0.0.0",
        "--server.headless",
        "true",
        "--browser.gatherUsageStats",
        "false",
    ],
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
)

time.sleep(4)
if streamlit_proc.poll() is not None:
    streamlit_log.close()
    print(LOG_FILE.read_text(errors="replace")[-3000:])
    raise RuntimeError("Streamlit stopped early. Check the log above.")

print("Streamlit is running.")
print("Open the forwarded port 8501 from the Codespaces Ports panel.")
print(f"Local service URL inside Codespaces: http://localhost:{PORT}")
print(f"Log file: {LOG_FILE}")


Streamlit is running locally.
Open this URL in your browser: http://localhost:8501


## 4. Stop the app

Run this only when you are done with the viewer.


In [ ]:
import subprocess

subprocess.run(["pkill", "-f", "streamlit run"], check=False)
print("Stopped Streamlit process.")
